# 🎬 B-Roll Editor
**Three steps:**
1. Run Cell 1 — installs everything (takes ~2 minutes, once per session)
2. Run Cell 2 — paste your Anthropic API key
3. Run Cell 3 — upload your video, wait, download the result

To run a cell: click it, then press the ▶ button on the left (or Shift+Enter)

In [ ]:
# ── CELL 1: Install everything ──────────────────────────────────────────────
print('Installing ffmpeg...')
import subprocess, sys
subprocess.run(['apt-get', 'install', '-y', 'ffmpeg'], capture_output=True)
print('Installing Python packages (this takes ~1 min)...')
subprocess.run([sys.executable, '-m', 'pip', 'install', '-q',
                'anthropic', 'faster-whisper', 'requests'], check=True)
print('\n✅ All done! Run the next cell.')

In [ ]:
# ── CELL 2: Your Anthropic API key ──────────────────────────────────────────
# Get yours free at https://console.anthropic.com → API Keys
ANTHROPIC_API_KEY = "paste-your-key-here"

if ANTHROPIC_API_KEY == "paste-your-key-here":
    print('⚠️  Replace paste-your-key-here with your actual key, then re-run this cell.')
else:
    print('✅ API key set!')

In [ ]:
# ── CELL 3: Upload video → process → download ───────────────────────────────
import os, json, subprocess, requests, base64, uuid, shutil
from google.colab import files

PEXELS_API_KEY = "LppRQMMFSN0E7avfYQUoeQVdifahsxZwB0Uzag6Z7OQhZvemwAYfQ5eu"

# ── Upload ──
print('📁 Choose your MP4 file...')
uploaded = files.upload()
video_filename = list(uploaded.keys())[0]
print(f'✅ Uploaded: {video_filename}')

# ── Helpers ──
def run_ff(*args, label='ffmpeg'):
    r = subprocess.run(['ffmpeg','-y']+list(args), capture_output=True, text=True)
    if r.returncode != 0:
        raise RuntimeError(f'{label} failed:\n{r.stderr[-1500:]}')

def get_info(path):
    r = subprocess.run(['ffprobe','-v','quiet','-print_format','json',
                        '-show_streams','-show_format', path],
                       capture_output=True, text=True, check=True)
    d = json.loads(r.stdout)
    w = h = None
    for s in d.get('streams',[]):
        if s.get('codec_type')=='video':
            w,h = s['width'],s['height']; break
    dur = float(d.get('format',{}).get('duration',0))
    return w, h, dur

def transcribe(audio_path, duration):
    from faster_whisper import WhisperModel
    print('  Loading Whisper model (~150 MB download on first run)...')
    model = WhisperModel('base', device='cpu', compute_type='int8')
    segs, _ = model.transcribe(audio_path, beam_size=5,
                                word_timestamps=True, vad_filter=True)
    transcript, chunk_start, chunk_words, chunk_end = [], None, [], 0.0
    for seg in segs:
        for w in (seg.words or []):
            if chunk_start is None: chunk_start = w.start
            chunk_words.append(w.word); chunk_end = w.end
            ends_sent = w.word.strip().endswith(('.','!','?',','))
            dur = chunk_end - chunk_start
            if ends_sent and dur >= 3.0 or dur >= 15.0:
                transcript.append({'start':round(chunk_start,2),'end':round(chunk_end,2),
                                   'text':''.join(chunk_words).strip()})
                chunk_start, chunk_words = None, []
    if chunk_words and chunk_start is not None:
        transcript.append({'start':round(chunk_start,2),'end':round(duration,2),
                           'text':''.join(chunk_words).strip()})
    elif transcript:
        transcript[-1]['end'] = round(duration,2)
    return transcript or [{'start':0.0,'end':duration,'text':'(no speech)'}]

def plan_edit(transcript, duration):
    import anthropic as ant
    client = ant.Anthropic(api_key=ANTHROPIC_API_KEY) \
        if ANTHROPIC_API_KEY.startswith('sk-ant-api') \
        else ant.Anthropic(auth_token=ANTHROPIC_API_KEY)
    lines = '\n'.join(f"[{s['start']:.1f}s-{s['end']:.1f}s]: {s['text']}" for s in transcript)
    resp = client.messages.create(
        model='claude-opus-4-5', max_tokens=4096,
        messages=[{'role':'user','content':(
            'You are a video editor. Plan this edit.\n\n'
            f'TRANSCRIPT:\n{lines}\n\n'
            'RULES:\n'
            '1. face_cam: hooks, CTAs, emotional moments, direct address\n'
            '2. broll: products, places, concepts, tips, steps\n'
            '3. Min 3s per segment, aim 40-60% broll\n'
            f'4. Cover 0.0 to {duration:.1f}s exactly\n'
            '5. broll keyword: 2-4 words, concrete, searchable\n'
            '6. First segment almost always face_cam\n\n'
            'Return ONLY a JSON array:\n'
            '[{"start":0.0,"end":9.0,"type":"face_cam"},\n'
            ' {"start":9.0,"end":18.0,"type":"broll","keyword":"coffee morning desk"},\n'
            f' {{"start":18.0,"end":{duration:.1f},"type":"face_cam"}}]\n'
            f'Last segment ends at exactly {duration:.1f}'
        )}])
    text = resp.content[0].text.strip()
    if text.startswith('```'):
        parts = text.split('```'); text = parts[1]
        if text.startswith('json'): text = text[4:]
    segs = json.loads(text.strip())
    if segs: segs[-1]['end'] = duration
    merged = [segs[0]]
    for s in segs[1:]:
        if s['end']-s['start'] < 3.0: merged[-1]['end'] = s['end']
        else: merged.append(s)
    return merged

def get_pexels(keyword):
    r = requests.get('https://api.pexels.com/videos/search',
                     headers={'Authorization': PEXELS_API_KEY},
                     params={'query':keyword,'orientation':'portrait','per_page':10}, timeout=30)
    for vid in r.json().get('videos',[]):
        for vf in sorted(vid.get('video_files',[]),key=lambda x:x.get('height',0),reverse=True):
            if vf.get('width',9999) < vf.get('height',0): return vf['link']
    vids = r.json().get('videos',[])
    if vids and vids[0].get('video_files'): return vids[0]['video_files'][0]['link']
    return None

def dl(url, dest):
    r = requests.get(url, stream=True, timeout=120)
    with open(dest,'wb') as f:
        for chunk in r.iter_content(65536): f.write(chunk)

# ── Pipeline ──
work = f'job_{uuid.uuid4().hex[:8]}'
os.makedirs(work)

print('\n[1/7] Reading video info...')
w, h, duration = get_info(video_filename)
print(f'      {w}x{h}, {duration:.1f}s')

print('[2/7] Extracting audio...')
audio = f'{work}/audio.mp3'
run_ff('-i',video_filename,'-vn','-acodec','libmp3lame','-ar','16000','-ac','1','-b:a','32k',audio)

print('[3/7] Transcribing (Whisper)...')
transcript = transcribe(audio, duration)
print(f'      {len(transcript)} segments transcribed')

print('[4/7] Planning edit with Claude...')
segments = plan_edit(transcript, duration)
broll_count = sum(1 for s in segments if s['type']=='broll')
broll_pct = sum(s['end']-s['start'] for s in segments if s['type']=='broll') / duration * 100
print(f'      {len(segments)} segments — {broll_count} b-roll ({broll_pct:.0f}% of video)')
print('      Edit plan:')
for s in segments:
    kw = f" [{s['keyword']}]" if s.get('keyword') else ''
    print(f"        {s['start']:.1f}s → {s['end']:.1f}s  {s['type']}{kw}")

print('[5/7] Downloading b-roll clips from Pexels...')
broll_clips = {}
broll_segs = [s for s in segments if s['type']=='broll']
for i, seg in enumerate(broll_segs):
    kw = seg.get('keyword','nature landscape')
    print(f'      Searching: "{kw}"...')
    url = get_pexels(kw)
    if url:
        p = f'{work}/broll_{i}.mp4'; dl(url, p); broll_clips[i] = p
        print(f'      ✓ Downloaded')
    else:
        broll_clips[i] = None; print(f'      ✗ Not found, will use face cam')

print('[6/7] Cutting and compositing...')
seg_files = []; broll_idx = 0
for i, seg in enumerate(segments):
    dur = seg['end'] - seg['start']
    out = f'{work}/seg_{i:03d}.mp4'
    if seg['type'] == 'face_cam':
        run_ff('-ss',str(seg['start']),'-i',video_filename,'-t',str(dur),
               '-vf',f'scale={w}:{h}:force_original_aspect_ratio=decrease,pad={w}:{h}:(ow-iw)/2:(oh-ih)/2:black',
               '-an','-c:v','libx264','-preset','fast','-crf','22',out)
    else:
        clip = broll_clips.get(broll_idx); broll_idx += 1
        if clip:
            run_ff('-stream_loop','-1','-i',clip,'-t',str(dur),
                   '-vf',f'scale={w}:{h}:force_original_aspect_ratio=increase,crop={w}:{h}',
                   '-an','-c:v','libx264','-preset','fast','-crf','22',out)
        else:
            run_ff('-ss',str(seg['start']),'-i',video_filename,'-t',str(dur),
                   '-vf',f'scale={w}:{h}:force_original_aspect_ratio=decrease,pad={w}:{h}:(ow-iw)/2:(oh-ih)/2:black',
                   '-an','-c:v','libx264','-preset','fast','-crf','22',out)
    seg_files.append(out)

concat = f'{work}/concat.txt'
with open(concat,'w') as f:
    for sp in seg_files: f.write(f"file '{os.path.abspath(sp)}'\n")
video_only = f'{work}/video_only.mp4'
run_ff('-f','concat','-safe','0','-i',concat,'-c:v','copy',video_only)
output = f'{work}/output.mp4'
run_ff('-i',video_only,'-i',video_filename,
       '-map','0:v:0','-map','1:a:0',
       '-c:v','copy','-c:a','aac','-b:a','192k','-shortest',output)

size_mb = os.path.getsize(output)/1024/1024
print(f'[7/7] Done! Output: {size_mb:.1f} MB')
print('\n⬇️  Downloading your video now...')
files.download(output)
shutil.rmtree(work)